In [19]:
# Semente fixa para reprodutibilidade
RANDOM_STATE = 42


# 02 — Pré-processamento e Engenharia de Atributos

**Grupo 50 • Tech Challenge — Fase 2**

## Objetivo

Integrar as bases, construir o `TARGET`, tratar duplicidades cadastrais, identificar perfis equivalentes, criar atributos derivados e preparar a tabela utilizada na modelagem.

O `credit_record` é utilizado para construção do alvo e não entra como variável preditora. O `PROFILE_GROUP` é criado a partir das características cadastrais e será utilizado exclusivamente para manter perfis equivalentes no mesmo conjunto durante a divisão treino/validação/teste.


In [20]:
from pathlib import Path
import pandas as pd
import numpy as np

# Localização robusta para execução no repositório local ou em ambiente de notebook.
candidatos = [
    Path.cwd().parent / "data" / "raw",
    Path.cwd() / "data" / "raw",
    Path("/content/data/raw"),
]

RAW = next(
    (p for p in candidatos if (p / "application_record.csv").exists()),
    None,
)

if RAW is None and Path("/content/application_record.csv").exists():
    RAW = Path("/content")

if RAW is None:
    raise FileNotFoundError(
        "Não foi possível localizar application_record.csv e credit_record.csv."
    )

ROOT = RAW.parent.parent if RAW.name == "raw" else RAW
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

application = pd.read_csv(RAW / "application_record.csv")
credit = pd.read_csv(RAW / "credit_record.csv")

print("Diretório das bases:", RAW)
print("application_record:", application.shape)
print("credit_record:", credit.shape)


Diretório das bases: c:\projetos\Grupo_50_Tech-Challenge-Fase-2\data\raw
application_record: (438557, 18)
credit_record: (1048575, 3)


## 1. Construção do TARGET

Para cada cliente, o histórico de crédito é convertido em uma classificação binária:

- `STATUS = 2, 3, 4 ou 5` → `TARGET = 1` (mau pagador);
- demais status → `TARGET = 0` (bom pagador).

O histórico de crédito participa somente da construção do alvo, evitando que o próprio histórico utilizado para definir o comportamento futuro seja utilizado como variável preditora.


In [21]:
# Indicador de ocorrência de status de inadimplência
status_maus = {"2", "3", "4", "5"}

credit_target = (
    credit.assign(
        MAU_PAGADOR=credit["STATUS"].astype(str).isin(status_maus).astype(int)
    )
    .groupby("ID")["MAU_PAGADOR"]
    .max()
    .rename("TARGET")
)

# Mantém uma única ocorrência cadastral por ID.
application_clean = application.drop_duplicates(
    subset="ID",
    keep="first",
).copy()

# Cruzamento das bases.
dados = application_clean.merge(
    credit_target,
    on="ID",
    how="inner",
)

print("Linhas cadastrais originais:", len(application))
print("IDs únicos após deduplicação:", application_clean["ID"].nunique())
print("Clientes com histórico de crédito:", credit["ID"].nunique())
print("Base final:", dados.shape)

distribuicao_target = (
    dados["TARGET"]
    .value_counts()
    .rename(index={0: "Bom pagador (0)", 1: "Mau pagador (1)"})
    .to_frame("quantidade")
)

distribuicao_target["percentual"] = (
    distribuicao_target["quantidade"] / len(dados) * 100
).round(2)

display(distribuicao_target)


Linhas cadastrais originais: 438557
IDs únicos após deduplicação: 438510
Clientes com histórico de crédito: 45985
Base final: (36457, 19)


,quantidade,percentual
TARGET,,
Bom pagador (0),35841,98.31
Mau pagador (1),616,1.69


## 2. Identificação dos perfis cadastrais

Registros com IDs diferentes podem representar exatamente o mesmo conjunto de características cadastrais.

Para evitar que essas cópias sejam distribuídas entre treino, validação e teste, o `PROFILE_GROUP` é criado **antes da imputação e da criação dos atributos derivados**, utilizando os atributos cadastrais originais, sem `ID` e sem `TARGET`.

Assim, o agrupamento representa o cadastro observado na base original e não incorpora estatísticas calculadas posteriormente.


In [22]:
# O perfil é definido pelas características cadastrais originais.
colunas_perfil = [
    c for c in application_clean.columns
    if c != "ID"
]

perfil_hash = pd.util.hash_pandas_object(
    dados[colunas_perfil].astype(str),
    index=False,
).astype(str)

dados["PROFILE_GROUP"] = perfil_hash

tamanho_perfil = dados.groupby("PROFILE_GROUP").size()

conflitos = (
    dados.groupby("PROFILE_GROUP")["TARGET"]
    .nunique()
    .gt(1)
)

# Número de clientes que pertencem a perfis que aparecem mais de uma vez.
clientes_perfis_repetidos = int(
    tamanho_perfil[tamanho_perfil > 1].sum()
)

print(
    "Perfis cadastrais distintos:",
    dados["PROFILE_GROUP"].nunique()
)
print(
    "Clientes em perfis repetidos:",
    clientes_perfis_repetidos
)
print(
    "Perfis com TARGET conflitante:",
    int(conflitos.sum())
)
print(
    "Maior tamanho de grupo:",
    int(tamanho_perfil.max())
)

Perfis cadastrais distintos: 9728
Clientes em perfis repetidos: 33320
Perfis com TARGET conflitante: 269
Maior tamanho de grupo: 51


## 3. Engenharia de atributos

São criados dois atributos derivados:

- `AGE_YEARS`: idade aproximada em anos;
- `EMPLOYED_YEARS`: tempo de emprego aproximado em anos.

O valor `365243` em `DAYS_EMPLOYED` é tratado como valor especial/ausente antes da conversão.


In [23]:
df_model = dados.copy()

# Idade aproximada em anos.
df_model["AGE_YEARS"] = (
    -df_model["DAYS_BIRTH"] / 365.25
).round(2)

# Tempo de emprego.
df_model["EMPLOYED_YEARS"] = (
    df_model["DAYS_EMPLOYED"]
    .where(df_model["DAYS_EMPLOYED"] < 365243, np.nan)
    .abs()
    .div(365.25)
    .round(2)
)

display(
    df_model[["AGE_YEARS", "EMPLOYED_YEARS"]].describe()
)


,AGE_YEARS,EMPLOYED_YEARS
count,36457.000000,30322.000000
mean,43.737717,7.243014
std,11.500502,6.454193
min,20.500000,0.050000
25%,34.120000,2.680000
50%,42.610000,5.450000
75%,53.220000,9.600000
max,68.860000,43.020000


## 4. Tratamento de valores ausentes

Os valores ausentes das variáveis categóricas são preenchidos pela moda da própria variável.

Nas variáveis numéricas, a mediana é utilizada como medida robusta para completar os valores ausentes.

Esse tratamento deixa a base pronta para a codificação das variáveis e para a etapa de modelagem.


In [24]:
# Identificação dos tipos antes da remoção das variáveis brutas.
colunas_categoricas = df_model.select_dtypes(include="object").columns.tolist()
colunas_numericas = df_model.select_dtypes(include=np.number).columns.tolist()

colunas_numericas = [
    c for c in colunas_numericas
    if c != "TARGET"
]

# Categóricas: preenchimento pela moda.
for coluna in colunas_categoricas:
    moda = df_model[coluna].mode(dropna=True)
    if not moda.empty:
        df_model[coluna] = df_model[coluna].fillna(moda.iloc[0])

# Numéricas: preenchimento pela mediana.
for coluna in colunas_numericas:
    df_model[coluna] = df_model[coluna].fillna(
        df_model[coluna].median()
    )

print(
    "Valores ausentes restantes:",
    int(df_model.isna().sum().sum())
)


Valores ausentes restantes: 0


## 5. Preparação da matriz de modelagem

`ID`, `DAYS_BIRTH` e `DAYS_EMPLOYED` são removidos das variáveis preditoras porque:

- `ID` é identificador;
- `AGE_YEARS` substitui `DAYS_BIRTH`;
- `EMPLOYED_YEARS` substitui `DAYS_EMPLOYED`.

O `PROFILE_GROUP` é preservado no arquivo para ser utilizado exclusivamente na divisão agrupada e não será utilizado como feature.


In [25]:
drop_cols = [
    "ID",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
]

df_model = df_model.drop(
    columns=[c for c in drop_cols if c in df_model.columns]
)

# Remove variáveis constantes.
feature_cols = [
    c for c in df_model.columns
    if c not in ["TARGET", "PROFILE_GROUP"]
]

constant_cols = [
    c for c in feature_cols
    if df_model[c].nunique(dropna=False) <= 1
]

df_model = df_model.drop(columns=constant_cols)

print(
    "Variáveis constantes removidas:",
    constant_cols if constant_cols else "nenhuma",
)
print("Dimensão final:", df_model.shape)


Variáveis constantes removidas: ['FLAG_MOBIL']
Dimensão final: (36457, 18)


## 6. Verificações finais

A tabela final mantém:

- `TARGET` como variável resposta;
- `PROFILE_GROUP` exclusivamente para o agrupamento;
- atributos cadastrais e derivados como variáveis disponíveis para a modelagem.

A codificação das variáveis categóricas será realizada no notebook de modelagem.


In [26]:
features_finais = df_model.drop(
    columns=["TARGET", "PROFILE_GROUP"]
)

print("TARGET nulos:", int(df_model["TARGET"].isna().sum()))
print("PROFILE_GROUP nulos:", int(df_model["PROFILE_GROUP"].isna().sum()))
print("Quantidade de grupos:", df_model["PROFILE_GROUP"].nunique())
print("Quantidade de features antes do encoding:", features_finais.shape[1])
print("Nulos nas features:", int(features_finais.isna().sum().sum()))

display(
    pd.DataFrame({
        "tipo": features_finais.dtypes.astype(str),
        "nulos": features_finais.isna().sum(),
        "n_unicos": features_finais.nunique(dropna=False),
    }).sort_values("nulos", ascending=False)
)


TARGET nulos: 0
PROFILE_GROUP nulos: 0
Quantidade de grupos: 9728
Quantidade de features antes do encoding: 16
Nulos nas features: 0


,tipo,nulos,n_unicos
CODE_GENDER,object,0,2
FLAG_OWN_CAR,object,0,2
FLAG_OWN_REALTY,object,0,2
CNT_CHILDREN,int64,0,9
AMT_INCOME_TOTAL,float64,0,265
NAME_INCOME_TYPE,object,0,5
NAME_EDUCATION_TYPE,object,0,5
NAME_FAMILY_STATUS,object,0,5
NAME_HOUSING_TYPE,object,0,6
FLAG_WORK_PHONE,int64,0,2


In [27]:
# Artefato local para os notebooks seguintes.
# data/processed/ está no .gitignore e não será versionado.
output_path = PROC / "base_modelagem.csv"

df_model.to_csv(
    output_path,
    index=False,
)

print("Base de modelagem salva em:", output_path)
print("Shape salvo:", df_model.shape)


Base de modelagem salva em: c:\projetos\Grupo_50_Tech-Challenge-Fase-2\data\processed\base_modelagem.csv
Shape salvo: (36457, 18)


## Conclusão

O notebook entrega uma base integrada e preparada para a modelagem, com o `TARGET` construído a partir do histórico de crédito, atributos derivados de idade e tempo de emprego e tratamento dos valores ausentes.

O `PROFILE_GROUP` é preservado exclusivamente para a divisão agrupada entre treino, validação e teste, evitando que perfis cadastrais equivalentes sejam distribuídos entre conjuntos diferentes.
